# IMU 센서 종류 진단 — raw accelerometer(중력 포함) vs linear acceleration(중력 제거)

**왜 보나:** gravity alignment 는 중력 벡터가 신호에 실려 있어야만 가능하다.
센서가 *linear acceleration*(펌웨어가 중력을 빼버린 값)이면 중력 벡터가 없어 원리적으로 불가능하다.

**판별 논리 (근거):**
| 신호 종류 | 정지/중앙값 |accel| | 운동 중 거동 |
|---|---|---|
| raw accel (중력 O) | **≈ 1g** (0이 아님) | 1g 주변에서 출렁 |
| linear accel (중력 X) | **≈ 0** | 0 주변에서 출렁 |

→ 정지/중앙값 magnitude 가 0 근처면 linear, 1g 근처면 raw accel.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.abspath('..')

# 필요한 IMU 컬럼만 로드 (samsung1 은 triceps+biceps 둘 다 보유)
SRC = pd.read_parquet(os.path.join(ROOT, 'data', 'samsung1.parquet'),
                      columns=['triceps_X','triceps_Y','triceps_Z',
                               'biceps_X','biceps_Y','biceps_Z',
                               'filename','timestamp','subject'])
TGT = pd.read_parquet(os.path.join(ROOT, 'data', 'samsung2.parquet'),
                      columns=['triceps_X','triceps_Y','triceps_Z',
                               'csv_filename_l','Index_Time','subject_id'])

# (label, df, IMU 3축, 세션컬럼, 시간컬럼, 피험자컬럼)
SPECS = [
    ('samsung1 · triceps', SRC, ['triceps_X','triceps_Y','triceps_Z'], 'filename', 'timestamp', 'subject'),
    ('samsung1 · biceps',  SRC, ['biceps_X','biceps_Y','biceps_Z'],   'filename', 'timestamp', 'subject'),
    ('samsung2 · triceps', TGT, ['triceps_X','triceps_Y','triceps_Z'], 'csv_filename_l', 'Index_Time', 'subject_id'),
]
print('loaded | samsung1:', SRC.shape, '| samsung2:', TGT.shape)

In [ ]:
REST_N = 100   # 세션 앞 N 샘플 = 정지(rest) 추정 구간


def session_rest_mags(df, cols, sess, time, n=REST_N):
    """세션별 시작 n샘플(정지 추정) 평균 |accel| 리스트."""
    out = []
    for _, g in df.groupby(sess, sort=False):
        a = g.sort_values(time)[cols].to_numpy(np.float64)[:n]
        if len(a):
            out.append(np.sqrt((a ** 2).sum(1)).mean())
    return np.array(out)


def gravity_direction(df, cols, sess):
    """세션 평균 accel→단위벡터→세션간 평균 = 전역 중력 방향(단위벡터)."""
    dirs = []
    for _, g in df.groupby(sess, sort=False):
        m = g[cols].to_numpy(np.float64).mean(0)
        nrm = np.linalg.norm(m)
        if nrm > 1e-9:
            dirs.append(m / nrm)
    G = np.mean(dirs, 0)
    return G / (np.linalg.norm(G) + 1e-12)


def diagnose(label, df, cols, sess, time, subj):
    A = df[cols].to_numpy(np.float64)
    mag = np.sqrt((A ** 2).sum(1))
    nz = mag[mag > 1e-9]
    med = float(np.median(nz))
    rest_med = float(np.median(session_rest_mags(df, cols, sess, time)))
    unit = 'milli-g (×1000)' if med > 50 else 'g (×1)'
    g_scale = 1000.0 if med > 50 else 1.0
    return {
        'dataset': label,
        '단위(추정)': unit,
        'median |accel|': round(med, 3),
        'rest |accel|': round(rest_med, 3),
        'rest / 1g': round(rest_med / g_scale, 3),     # ≈1 → 중력, ≈0 → linear
        'linear이면 기대': '≈ 0',
        '판정': 'raw accel (중력 O)' if rest_med / g_scale > 0.3 else 'linear (중력 X)',
    }


verdict = pd.DataFrame([diagnose(*s) for s in SPECS])
print('=== [표1] 센서 종류 판정 — rest/median |accel| 이 0이 아니라 ~1g 이면 중력 존재 ===')
verdict

In [ ]:
# [표2] 축별 통계 + 중력 방향(DC). 정지 시 한 방향으로 쏠린 DC offset 자체가 중력의 흔적.
rows = []
for label, df, cols, sess, time, subj in SPECS:
    A = df[cols].to_numpy(np.float64)
    g = gravity_direction(df, cols, sess)
    rows.append({
        'dataset': label,
        'axis mean': np.round(A.mean(0), 3).tolist(),
        'axis std':  np.round(A.std(0), 3).tolist(),
        '중력방향(단위벡터)': np.round(g, 3).tolist(),
        'zero-pad %': round(100 * (np.sqrt((A ** 2).sum(1)) < 1e-9).mean(), 2),
    })
print('=== [표2] 축별 통계 & 추정 중력 방향 (방향이 잘 정의됨 → 정렬 기준 존재) ===')
pd.DataFrame(rows)

In [ ]:
# [표3] samsung1 피험자별 정지 |accel| — sub1 은 ~1g 깨끗, sub2 는 낮음(주의점 근거)
label, df, cols, sess, time, subj = SPECS[0]   # samsung1 triceps
rows = []
for sid, gsub in df.groupby(subj, sort=False):
    rm = session_rest_mags(gsub, cols, sess, time)
    rows.append({'subject': sid, 'n_sessions': len(rm),
                 'rest |accel| median': round(float(np.median(rm)), 3),
                 'rest |accel| min~max': f'{rm.min():.2f} ~ {rm.max():.2f}'})
print('=== [표3] samsung1 triceps 피험자별 정지 magnitude (samsung1 중력추정 노이즈 근거) ===')
pd.DataFrame(rows)

In [ ]:
# 시각화: |accel| 분포를 각자 1g 로 정규화해 겹쳐보기.
#   linear 라면 질량이 x=0 에 몰려야 함. 실제론 x=1(중력) 부근에 몰림.
fig, ax = plt.subplots(figsize=(9, 4.5))
for label, df, cols, sess, time, subj in SPECS:
    A = df[cols].to_numpy(np.float64)
    mag = np.sqrt((A ** 2).sum(1))
    mag = mag[mag > 1e-9]
    g_scale = 1000.0 if np.median(mag) > 50 else 1.0
    ax.hist(mag / g_scale, bins=200, range=(0, 2.5), histtype='step', density=True, label=label, linewidth=1.6)
ax.axvline(0, color='red', ls='--', lw=1.3, label='linear 기대값 (중력 X)')
ax.axvline(1, color='green', ls='--', lw=1.3, label='raw accel 기대값 (1g)')
ax.set_xlabel('|accel| / 1g'); ax.set_ylabel('density')
ax.set_title('IMU magnitude 분포 — 질량이 0이 아닌 1g 부근에 몰림 → 중력 존재(raw accel)')
ax.legend(); plt.tight_layout(); plt.show()

## 결론 (근거 요약)

- **[표1]** rest·median `|accel|` 이 0이 아니라 **각 단위 기준 ~1g** (samsung2 ≈ 1000 milli-g, samsung1 sub1 ≈ 1.0 g).
  linear accel이면 0 근처여야 하므로 → **중력 포함 raw accelerometer**.
- **[표2]** 축별 평균에 한 방향으로 쏠린 DC offset 이 존재하고, 추정 중력 방향이 안정적으로 정의됨 → **정렬 기준축 존재**.
- **[그림]** magnitude 질량이 `x=0`(linear 기대)이 아니라 `x≈1`(1g)에 몰림.

→ **gravity alignment 가능.** 단 주의: ① 단위가 samsung1=g / samsung2=milli-g 로 1000배 차이(방향 정렬·z-score 라 무해), ② **[표3]** samsung1 은 피험자별 정지 magnitude 가 들쭉날쭉(sub2 가 낮음) → source 중력 추정에 노이즈.